<a href="https://colab.research.google.com/github/Tiff-Maina/Knowledge-Engineering/blob/main/Autonomous_Trailer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ICS 3202: Artificial Intelligence
## Practical Exercise 4: Developing a Knowledge-Based Agent

---

### Group Members

| # | Name | Registration Number |
|---|------|---------------------|
| 1 | *Fabunmi, Eniola Faithfulness* | *167925* |
| 2 | *Maina, Tiffany Wanjiru* | *189592* |
| 3 | *Ogwayo, Emma Awuor* | *189923* |

### 1. Exercise Overview

In Practical Exercise 3, we designed a self-driving trailer agent using the PEAS framework, represented Kenya's highway network as a directed graph, and identified the optimal route from Nairobi to Kisumu. In the subsequent extension, we implemented Uniform Cost Search (UCS) in Python to determine the shortest route, obtaining an optimal path of 347 km.

In this exercise, we extend the same system by developing a **knowledge-based agent**. Instead of relying only on a predefined graph, the trailer will use a knowledge base containing information about towns, roads, distances and regions.

The agent will apply inference rules to derive additional knowledge, answer queries about road connections and reachability, identify towns without outgoing roads, and determine which movements are permitted. Uniform Cost Search will then use the knowledge base to plan the lowest-cost route from Nairobi to Kisumu.

The implementation demonstrates how a knowledge-based system combines stored information, logical reasoning and search to make decisions.

## 2. Agent Design: PEAS Framework

The self-driving trailer agent is designed to transport cargo safely from Nairobi to Kisumu using Kenya's highway network. The PEAS framework defines the agent's performance measures, environment, actuators and sensors.

| Component | Description |
|---|---|
| **Performance Measure** | Deliver cargo safely and legally to Kisumu, reach the correct destination, minimise the total travelling distance and fuel consumption, and reduce travel time while obeying traffic laws. |
| **Environment** | Kenya's highway network covering Nairobi, Central Highlands, Rift Valley and Lake Victoria Basin. It also includes road infrastructure, other road users, pedestrians, animals and changing weather conditions. |
| **Actuators** | Power steering, accelerator, braking system, horn, hazard lights, indicators, headlights, windshield wipers and onboard display. |
| **Sensors** | GPS receiver, cameras and computer vision, radar, speed sensors and distance sensors. |

The PEAS framework helps define the information the trailer needs, the actions it can perform and the objectives it must achieve during its journey.

## 3. Environment Classification

The self-driving trailer operates in an environment with the following characteristics:

| Property | Classification | Explanation |
|---|---|---|
| **Observability** | Partially observable | The trailer cannot observe the entire highway network at once. For example, while travelling from Nairobi, it may not have complete information about traffic conditions in Nakuru. |
| **Determinism** | Stochastic | Outcomes may change due to unpredictable events such as other drivers, pedestrians, animals and weather conditions. |
| **Episodic / Sequential** | Sequential | Each movement affects the trailer's current location and the possible routes available for subsequent decisions. |
| **Static / Dynamic** | Dynamic | Road conditions, traffic lights, pedestrians and other vehicles can change while the trailer is operating. |
| **Agents** | Multi-agent | The trailer shares the road with other drivers, pedestrians and road users whose actions can affect its journey. |

The sequential nature of the environment is particularly important because the trailer must consider how its current decisions affect future movements. This makes a knowledge base useful for storing road information and deriving possible routes before selecting an action.

## 4. Knowledge Base Design

### 4.1 Design Objective

The knowledge base stores information about Kenya's highway network and allows the trailer to reason about its available movements.

It is designed to answer three main categories of questions:

1. **Road connections:** Which towns are directly connected by roads?
2. **Reachability:** Can the trailer travel from one town to another through one or more roads? Which towns have no outgoing roads?
3. **Route planning:** What is the lowest-cost route from Nairobi to Kisumu?

The knowledge base handles the storage of facts and logical reasoning, while Uniform Cost Search handles route optimisation. This separates determining what movements are possible from determining which available route has the lowest cost.

### 4.2 Knowledge Representation

The system uses predicates to represent information about the highway network.

| Predicate | Arguments | Meaning |
|---|---|---|
| `road(X,Y,D)` | 3 | A directed road exists from town X to town Y, with distance D kilometres. |
| `region(X,R)` | 2 | Town X belongs to region R. |
| `town(X)` | 1 | X is a town in the highway network. |
| `connected(X,Y)` | 2 | A direct road exists from X to Y. |
| `reachable(X,Y)` | 2 | A route consisting of one or more roads exists from X to Y. |
| `dead_end(X)` | 1 | Town X has no outgoing roads. |
| `legal_move(X,Y)` | 2 | Travelling directly from X to Y is a permitted movement in the network. |

The predicates `road` and `region` represent the initial facts supplied to the knowledge base. The remaining predicates are derived through inference rules.

Reachability is non-reflexive: a town is not considered reachable from itself unless a path containing one or more roads exists.

### 4.3 Asserted Knowledge

The initial knowledge base contains information from the directed graph developed in Practical Exercise 3 and used in the UCS extension.

The network contains **40 directed road connections across 29 towns**. Each road is represented by its starting town, destination town and distance in kilometres.

The directed nature of the graph is preserved. A road from town X to town Y does not automatically imply that a road exists from Y to X.

The road connections and their distances are encoded as `road` facts in the implementation section.

#### Region Facts

The network is also organised into the four geographical regions used in Practical Exercise 3.

| Region | Towns |
|---|---|
| Nairobi | Nairobi, Ruiru, Thika |
| Central Highlands | Nyeri, Nanyuki, Nyahururu, Embu, Chuka, Meru, Timau, Isiolo |
| Rift Valley | Naivasha, Nakuru, Njoro, Molo, Eldama Ravine, Kabarnet, Eldoret, Kapsabet, Narok, Bomet, Olenguruone, Kericho, Webuye, Kakamega |
| Lake Victoria Basin | Kisumu, Kisii, Nyamira, Kilgoris |

These geographical classifications are stored using the `region` predicate. They provide additional information about the towns in the network, although they are not required for the reachability inference rules.

### 4.4 Inference Rules

The knowledge base uses six inference rules to derive new information from the initially asserted facts.

| Rule | Premise | Conclusion |
|---|---|---|
| **R1: Town Identification** | `road(X,Y,D)` | `town(X)` and `town(Y)` |
| **R2: Direct Connection** | `road(X,Y,D)` | `connected(X,Y)` |
| **R3: Initial Reachability** | `connected(X,Y)` | `reachable(X,Y)` |
| **R4: Transitive Reachability** | `reachable(X,Z)` and `connected(Z,Y)` | `reachable(X,Y)` |
| **R5: Dead-End Identification** | `town(X)` and no outgoing `road` from X | `dead_end(X)` |
| **R6: Legal Movement** | `road(X,Y,D)` | `legal_move(X,Y)` |

**Explanation of the rules**

- R1 identifies the towns involved in the road network from the road facts.
- R2 derives direct connections between towns.
- R3 establishes that a directly connected town is reachable.
- R4 derives additional reachable towns by combining existing paths with direct connections. This allows the agent to identify routes involving multiple road segments.
- R5 identifies towns that have no outgoing road connections.
- R6 identifies the direct movements available to the trailer.

R4 is particularly important because it allows the agent to establish reachability without storing every possible route as an initial fact.   

R5 also demonstrates inference because dead-end information is derived by examining road facts rather than being manually entered.   

Under this definition Kisumu is identified as a dead end despite being the successful destination of the journey.

### 4.5 Inference Strategy: Forward Chaining

The knowledge base uses **forward chaining** to derive additional information.It is a data-driven inference technique that starts with known facts and repeatedly applies inference rules until no additional facts can be derived. This final state is known as a fixed point.

For this implementation, the inference engine applies R1–R6 repeatedly, adding newly derived facts while ignoring duplicates.

Forward chaining is suitable because the trailer needs to answer several queries during operation — direct connections, reachability and legal movements — and deriving this information in advance lets each query be answered directly from the knowledge base.

### 4.6 Integration with Uniform Cost Search

The knowledge base determines which movements are available and which towns are reachable. However, it does not independently determine the cheapest route.

Uniform Cost Search is therefore used as the planning algorithm. It obtains legal movements and their corresponding road distances from the knowledge base, then uses a priority queue to explore paths in increasing order of accumulated cost.

The goal test is performed when Kisumu is removed from the priority queue. This delayed goal test ensures that the returned solution has the lowest total path cost when road distances are non-negative.

### 4.7 Agent Decision Cycle

The trailer follows the following decision cycle:

1. Obtain its current location.
2. Query the knowledge base for available legal movements.
3. Check whether the destination is reachable.
4. Use Uniform Cost Search to determine the lowest-cost route.
5. Select the next town along the planned route.
6. Update its current location and repeat until Kisumu is reached.

The implementation uses a simulated Nairobi-to-Kisumu journey to demonstrate this decision cycle without requiring live vehicle sensors or physical actuators.


## 5. Knowledge Base Implementation

We implemented a knowledge base in Python to store information about towns, road connections and geographical regions. The initial facts are added to the knowledge base and used by the inference engine in the next section.


In [7]:
# ============================================================
# Section 5 — Knowledge Base Implementation
# ============================================================

class KnowledgeBase:
    """Set-based knowledge base for storing and querying facts."""

    def __init__(self):
        self.facts = set()
        self.derived_log = []

    # Add an initial fact
    def tell(self, fact):
        if fact in self.facts:
            return False

        self.facts.add(fact)
        return True

    # Add a fact produced through inference
    def tell_derived(self, fact):
        if fact in self.facts:
            return False

        self.facts.add(fact)
        self.derived_log.append(fact)
        return True

    # Retrieve facts matching a predicate and its arguments
    def ask(self, predicate, *args):
        results = []

        for fact in self.facts:
            if fact[0] != predicate:
                continue

            if len(fact) - 1 != len(args):
                continue

            if all(
                expected is None or expected == actual
                for expected, actual in zip(args, fact[1:])
            ):
                results.append(fact)

        return sorted(results, key=repr)

    # Check whether a particular fact exists
    def __contains__(self, fact):
        return fact in self.facts

    # Count facts belonging to each predicate
    def summary(self):
        counts = {}

        for fact in self.facts:
            predicate = fact[0]
            counts[predicate] = counts.get(predicate, 0) + 1

        return counts


# ------------------------------------------------------------
# Base Knowledge: Directed Road Connections
# ------------------------------------------------------------

ROAD_FACTS = [
    ('Nairobi', 'Ruiru', 26),
    ('Nairobi', 'Naivasha', 88),
    ('Nairobi', 'Narok', 142),
    ('Ruiru', 'Thika', 20),
    ('Thika', 'Nyeri', 105),
    ('Thika', 'Embu', 95),
    ('Embu', 'Chuka', 65),
    ('Chuka', 'Meru', 64),
    ('Meru', 'Timau', 32),
    ('Nyeri', 'Nanyuki', 62),
    ('Nyeri', 'Nyahururu', 104),
    ('Nanyuki', 'Timau', 22),
    ('Timau', 'Isiolo', 56),
    ('Nyahururu', 'Nakuru', 64),
    ('Naivasha', 'Nakuru', 69),
    ('Naivasha', 'Nyahururu', 98),
    ('Nakuru', 'Njoro', 24),
    ('Nakuru', 'Molo', 47),
    ('Nakuru', 'Kericho', 107),
    ('Nakuru', 'Eldama Ravine', 63),
    ('Nakuru', 'Eldoret', 156),
    ('Njoro', 'Molo', 35),
    ('Molo', 'Kericho', 64),
    ('Eldama Ravine', 'Kabarnet', 65),
    ('Eldama Ravine', 'Eldoret', 103),
    ('Eldoret', 'Kapsabet', 46),
    ('Eldoret', 'Webuye', 65),
    ('Kapsabet', 'Kisumu', 87),
    ('Webuye', 'Kakamega', 42),
    ('Kakamega', 'Kisumu', 50),
    ('Narok', 'Bomet', 82),
    ('Bomet', 'Olenguruone', 52),
    ('Bomet', 'Kericho', 77),
    ('Bomet', 'Kisii', 83),
    ('Olenguruone', 'Molo', 48),
    ('Kisii', 'Kilgoris', 62),
    ('Kisii', 'Nyamira', 27),
    ('Kisii', 'Kisumu', 120),
    ('Nyamira', 'Kericho', 58),
    ('Kericho', 'Kisumu', 83)
]


# ------------------------------------------------------------
# Base Knowledge: Geographical Regions
# ------------------------------------------------------------

REGION_FACTS = {
    'Nairobi': [
        'Nairobi', 'Ruiru', 'Thika'
    ],

    'Central Highlands': [
        'Nyeri', 'Nanyuki', 'Nyahururu', 'Embu',
        'Chuka', 'Meru', 'Timau', 'Isiolo'
    ],

    'Rift Valley': [
        'Naivasha', 'Nakuru', 'Njoro', 'Molo',
        'Eldama Ravine', 'Kabarnet', 'Eldoret',
        'Kapsabet', 'Narok', 'Bomet', 'Olenguruone',
        'Kericho', 'Webuye', 'Kakamega'
    ],

    'Lake Victoria Basin': [
        'Kisumu', 'Kisii', 'Nyamira', 'Kilgoris'
    ]
}


# ------------------------------------------------------------
# Build the Knowledge Base
# ------------------------------------------------------------

kb = KnowledgeBase()

# Assert road facts
for source, destination, distance in ROAD_FACTS:
    kb.tell(('road', source, destination, distance))

# Assert region facts
for region, towns in REGION_FACTS.items():
    for town in towns:
        kb.tell(('region', town, region))


# ------------------------------------------------------------
# Display Initial Knowledge Base
# ------------------------------------------------------------

print("=" * 55)
print("INITIAL KNOWLEDGE BASE")
print("=" * 55)

print(f"Total asserted facts : {len(kb.facts)}")
print(f"Road connections    : {len(kb.ask('road', None, None, None))}")
print(f"Region facts        : {len(kb.ask('region', None, None))}")
print(f"Distinct towns      : {len({
    town
    for fact in ROAD_FACTS
    for town in fact[:2]
})}")

INITIAL KNOWLEDGE BASE
Total asserted facts : 69
Road connections    : 40
Region facts        : 29
Distinct towns      : 29



## 6. Inference Engine (Forward Chaining)

We implemented a forward-chaining inference engine to apply the six rules defined in Section 4.4. It repeatedly derives new facts from the existing knowledge base until no additional facts can be generated.  

The engine converged after **6 productive passes**; the 7th pass confirmed the fixed point by deriving 0 new facts. The knowledge base grew from 69 asserted facts to **353 total facts** (284 derived via inference).

In [8]:
# ============================================================
# Section 6 — Inference Engine (Forward Chaining)
# ============================================================

def forward_chain(kb, verbose=True):
    """Apply R1-R6 until no additional facts can be derived."""

    pass_number = 0

    while True:
        pass_number += 1
        newly_derived = 0

        # Snapshot of the asserted road facts
        roads = kb.ask('road', None, None, None)

        # ----------------------------------------------------
        # R1: road(X,Y,D) -> town(X), town(Y)
        # ----------------------------------------------------

        for fact in roads:
            newly_derived += kb.tell_derived(
                ('town', fact[1])
            )

            newly_derived += kb.tell_derived(
                ('town', fact[2])
            )

        # ----------------------------------------------------
        # R2: road(X,Y,D) -> connected(X,Y)
        # ----------------------------------------------------

        for fact in roads:
            newly_derived += kb.tell_derived(
                ('connected', fact[1], fact[2])
            )

        # ----------------------------------------------------
        # R3: connected(X,Y) -> reachable(X,Y)
        # ----------------------------------------------------

        connections = kb.ask('connected', None, None)

        for fact in connections:
            newly_derived += kb.tell_derived(
                ('reachable', fact[1], fact[2])
            )

        # ----------------------------------------------------
        # R4: reachable(X,Z) AND connected(Z,Y)
        #     -> reachable(X,Y)
        # ----------------------------------------------------

        reachable_facts = kb.ask('reachable', None, None)

        for reachable in reachable_facts:
            x, z = reachable[1], reachable[2]

            for connection in connections:
                if connection[1] == z:
                    y = connection[2]

                    newly_derived += kb.tell_derived(
                        ('reachable', x, y)
                    )

        # ----------------------------------------------------
        # R5: town(X) with no outgoing road -> dead_end(X)
        # ----------------------------------------------------

        towns_with_roads = {
            fact[1] for fact in roads
        }

        for fact in kb.ask('town', None):
            town = fact[1]

            if town not in towns_with_roads:
                newly_derived += kb.tell_derived(
                    ('dead_end', town)
                )

        # ----------------------------------------------------
        # R6: road(X,Y,D) -> legal_move(X,Y)
        # ----------------------------------------------------

        for fact in roads:
            newly_derived += kb.tell_derived(
                ('legal_move', fact[1], fact[2])
            )

        if verbose:
            print(
                f"Pass {pass_number}: "
                f"{newly_derived} new facts derived"
            )

        # Stop when no new facts are generated
        if newly_derived == 0:
            break

    return pass_number


# ------------------------------------------------------------
# Execute Forward Chaining
# ------------------------------------------------------------

print("=" * 60)
print("FORWARD CHAINING INFERENCE ENGINE")
print("=" * 60)

total_passes = forward_chain(kb)

print("-" * 60)
print(f"Inference completed after {total_passes} passes.")


# ------------------------------------------------------------
# Display Knowledge Base After Inference
# ------------------------------------------------------------

print("\nKNOWLEDGE BASE SUMMARY")
print("-" * 60)

print(f"Total facts: {len(kb.facts)}")

for predicate, count in sorted(kb.summary().items()):
    print(f"{predicate:<12}: {count}")


# ------------------------------------------------------------
# Sample Queries
# ------------------------------------------------------------

print("\nSAMPLE KNOWLEDGE BASE QUERIES")
print("-" * 60)

checks = [
    (
        "Is Kisumu reachable from Nairobi?",
        ('reachable', 'Nairobi', 'Kisumu'),
        True
    ),
    (
        "Is Meru reachable from Nairobi?",
        ('reachable', 'Nairobi', 'Meru'),
        True
    ),
    (
        "Is Nairobi reachable from Kisumu?",
        ('reachable', 'Kisumu', 'Nairobi'),
        False
    ),
    (
        "Is Kilgoris a dead end?",
        ('dead_end', 'Kilgoris'),
        True
    ),
    (
        "Is Nairobi a dead end?",
        ('dead_end', 'Nairobi'),
        False
    ),
    (
        "Does Kisumu have no outgoing roads?",
        ('dead_end', 'Kisumu'),
        True
    )
]

for question, fact, expected in checks:
    result = fact in kb

    status = "OK" if result == expected else "FAILED"
    answer = "Yes" if result else "No"

    print(f"{question:<43} {answer:<5} [{status}]")


# ------------------------------------------------------------
# Demonstrate Derived Reachability
# ------------------------------------------------------------

print("\nREACHABILITY DEMONSTRATION")
print("-" * 60)

nairobi_reachable = kb.ask(
    'reachable', 'Nairobi', None
)

print("Derived fact: reachable(Nairobi, Kisumu)")
print(
    f"Present in knowledge base: "
    f"{('reachable', 'Nairobi', 'Kisumu') in kb}"
)

print(
    f"Towns reachable from Nairobi: "
    f"{len(nairobi_reachable)} of 29"
)

print("\nTowns reachable from Nairobi:")

for fact in nairobi_reachable:
    print(f"  {fact[2]}")

FORWARD CHAINING INFERENCE ENGINE
Pass 1: 199 new facts derived
Pass 2: 40 new facts derived
Pass 3: 24 new facts derived
Pass 4: 14 new facts derived
Pass 5: 6 new facts derived
Pass 6: 1 new facts derived
Pass 7: 0 new facts derived
------------------------------------------------------------
Inference completed after 7 passes.

KNOWLEDGE BASE SUMMARY
------------------------------------------------------------
Total facts: 353
connected   : 40
dead_end    : 4
legal_move  : 40
reachable   : 171
region      : 29
road        : 40
town        : 29

SAMPLE KNOWLEDGE BASE QUERIES
------------------------------------------------------------
Is Kisumu reachable from Nairobi?           Yes   [OK]
Is Meru reachable from Nairobi?             Yes   [OK]
Is Nairobi reachable from Kisumu?           No    [OK]
Is Kilgoris a dead end?                     Yes   [OK]
Is Nairobi a dead end?                      No    [OK]
Does Kisumu have no outgoing roads?         Yes   [OK]

REACHABILITY DEMONSTRATI


## 7. UCS Planner

We implemented Uniform Cost Search using the knowledge base to retrieve legal movements and their corresponding road distances. The algorithm uses a priority queue to find the route with the lowest accumulated distance from Nairobi to Kisumu.  
The planner retrieved all 40 legal road connections from the knowledge base, confirming that UCS operates entirely on inferred facts rather than a hard-coded graph.


In [9]:

# UCS Planner.
import heapq

# Retrieve legal destinations and their road distances from the knowledge base
def get_legal_neighbors(kb, town):
    neighbors = []

    legal_moves = kb.ask('legal_move', town, None)

    for move in legal_moves:
        destination = move[2]

        # Retrieve the distance of the corresponding road
        road_facts = kb.ask('road', town, destination, None)

        if road_facts:
            distance = road_facts[0][3]
            neighbors.append((destination, distance))

    return neighbors


# Uniform Cost Search using information stored in the knowledge base
def uniform_cost_search_kb(kb, start, goal):
    priority_queue = [(0, start, [start])]
    visited = set()
    traversal = []

    while priority_queue:
        cost, current_town, path = heapq.heappop(priority_queue)

        if current_town in visited:
            continue

        visited.add(current_town)
        traversal.append((current_town, cost, path))

        # Goal test when the town is removed from the priority queue
        if current_town == goal:
            return cost, path, traversal

        # Expand only legal movements retrieved from the knowledge base
        for neighbor, distance in get_legal_neighbors(kb, current_town):
            if neighbor not in visited:
                new_cost = cost + distance
                new_path = path + [neighbor]

                heapq.heappush(
                    priority_queue,
                    (new_cost, neighbor, new_path)
                )

    return float('inf'), [], traversal


# Check the number of legal road connections available to the planner
town_facts = kb.ask('town', None)
legal_connections = sum(
    len(get_legal_neighbors(kb, fact[1]))
    for fact in town_facts
)

print("=" * 60)
print("KNOWLEDGE-BASED UCS PLANNER")
print("=" * 60)
print(f"Legal road connections retrieved: {legal_connections}")
print(f"Legal moves from Nairobi: {get_legal_neighbors(kb, 'Nairobi')}")

# Test the planner
start_town = 'Nairobi'
goal_town = 'Kisumu'

if kb.ask('reachable', start_town, goal_town):
    optimal_cost, optimal_path, traversal = uniform_cost_search_kb(
        kb, start_town, goal_town
    )

    print("\nUCS PLANNING RESULT")
    print("-" * 60)
    print(f"Starting town : {start_town}")
    print(f"Destination   : {goal_town}")
    print(f"Optimal route : {' -> '.join(optimal_path)}")
    print(f"Total distance: {optimal_cost} km")
    print(f"Towns explored: {len(traversal)}")
else:
    print(f"No route exists from {start_town} to {goal_town}.")


KNOWLEDGE-BASED UCS PLANNER
Legal road connections retrieved: 40
Legal moves from Nairobi: [('Naivasha', 88), ('Narok', 142), ('Ruiru', 26)]

UCS PLANNING RESULT
------------------------------------------------------------
Starting town : Nairobi
Destination   : Kisumu
Optimal route : Nairobi -> Naivasha -> Nakuru -> Kericho -> Kisumu
Total distance: 347 km
Towns explored: 25



## 8. Self-Driving Trailer Agent

We implemented an agent that follows the route generated by UCS. At each step, it checks the knowledge base to confirm that the next movement is legal before updating its current location.


In [10]:
# Self-Driving Trailer Agent


def self_driving_trailer_agent(kb, start, goal):
    # Plan the route using UCS
    total_cost, path, traversal = uniform_cost_search_kb(kb, start, goal)

    if not path:
        print("No route available.")
        return

    current_town = start
    travelled = 0

    print("=" * 60)
    print("SELF-DRIVING TRAILER AGENT")
    print("=" * 60)
    print(f"Starting point: {start}")
    print(f"Destination  : {goal}")
    print("-" * 60)

    # Follow the planned route one movement at a time
    for next_town in path[1:]:
        legal_move = kb.ask('legal_move', current_town, next_town)
        road = kb.ask('road', current_town, next_town, None)

        if legal_move and road:
            distance = road[0][3]
            travelled += distance

            print(f"Move: {current_town} -> {next_town} ({distance} km)")
            current_town = next_town
        else:
            print(f"Movement not permitted: {current_town} -> {next_town}")
            return

    print("-" * 60)
    print(f"Current location: {current_town}")
    print(f"Total distance  : {travelled} km")
    print("Status          : Destination reached")


# Run the agent
self_driving_trailer_agent(kb, 'Nairobi', 'Kisumu')


SELF-DRIVING TRAILER AGENT
Starting point: Nairobi
Destination  : Kisumu
------------------------------------------------------------
Move: Nairobi -> Naivasha (88 km)
Move: Naivasha -> Nakuru (69 km)
Move: Nakuru -> Kericho (107 km)
Move: Kericho -> Kisumu (83 km)
------------------------------------------------------------
Current location: Kisumu
Total distance  : 347 km
Status          : Destination reached



## 9. Search Traversal

We displayed the order in which UCS explores the towns before reaching the destination. The traversal shows the accumulated cost and path at each expansion step.


In [11]:
# Search Traversal

optimal_cost, optimal_path, traversal = uniform_cost_search_kb(kb, 'Nairobi', 'Kisumu')

print("=" * 80)
print("UCS SEARCH TRAVERSAL")
print("=" * 80)

for step, (town, cost, path) in enumerate(traversal, start=1):
    path_str = ' -> '.join(path)
    print(f"Step {step:<2} | Town: {town:<15} | Cost: {cost:<3} km | Path: {path_str}")

print("-" * 80)
print(f"Optimal route: {' -> '.join(optimal_path)}")
print(f"Total distance: {optimal_cost} km")


UCS SEARCH TRAVERSAL
Step 1  | Town: Nairobi         | Cost: 0   km | Path: Nairobi
Step 2  | Town: Ruiru           | Cost: 26  km | Path: Nairobi -> Ruiru
Step 3  | Town: Thika           | Cost: 46  km | Path: Nairobi -> Ruiru -> Thika
Step 4  | Town: Naivasha        | Cost: 88  km | Path: Nairobi -> Naivasha
Step 5  | Town: Embu            | Cost: 141 km | Path: Nairobi -> Ruiru -> Thika -> Embu
Step 6  | Town: Narok           | Cost: 142 km | Path: Nairobi -> Narok
Step 7  | Town: Nyeri           | Cost: 151 km | Path: Nairobi -> Ruiru -> Thika -> Nyeri
Step 8  | Town: Nakuru          | Cost: 157 km | Path: Nairobi -> Naivasha -> Nakuru
Step 9  | Town: Njoro           | Cost: 181 km | Path: Nairobi -> Naivasha -> Nakuru -> Njoro
Step 10 | Town: Nyahururu       | Cost: 186 km | Path: Nairobi -> Naivasha -> Nyahururu
Step 11 | Town: Molo            | Cost: 204 km | Path: Nairobi -> Naivasha -> Nakuru -> Molo
Step 12 | Town: Chuka           | Cost: 206 km | Path: Nairobi -> Ruiru -> Th


## 10. Results and Discussion

The knowledge base successfully stored the highway connections and region information. Forward chaining derived additional facts, including legal movements, connected towns and reachable destinations.

The UCS planner used these facts to find the optimal route from Nairobi to Kisumu. The resulting route was:

**Nairobi → Naivasha → Nakuru → Kericho → Kisumu**

The total distance was **347 km**. The trailer agent then verified each movement against the knowledge base and successfully reached Kisumu.

This demonstrates how a knowledge-based agent can use stored information and inference to support route planning and decision-making.



## 11. Conclusion

This exercise demonstrated the development of a knowledge-based agent for a self-driving trailer. We represented road information as facts, applied forward chaining to derive new knowledge and integrated the knowledge base with Uniform Cost Search.

The agent successfully identified and followed the shortest-distance route from Nairobi to Kisumu, covering 347 km.
